# Day08：Dependency Graph

## Goal

基于 Day07 工程上下文建立可查询的 Unity 项目依赖图，验证直接依赖、反向依赖和传递依赖。活跃运行时代码继续保存在 `day06`。

## Setup

默认读取 `D:\Unity\Unity_Project\CodingAgentTest`。扫描和建图只读取 Unity 工程，因此 Unity 编辑器可以保持打开。

In [1]:
import json
import os
import sys
from pathlib import Path

workspace = Path.cwd().resolve()
if workspace.name == 'day08':
    workspace = workspace.parent
day06_path = workspace / 'day06'
if not day06_path.is_dir():
    raise RuntimeError(f'未找到 day06 源码目录: {day06_path}')
if str(day06_path) not in sys.path:
    sys.path.insert(0, str(day06_path))

unity_project_path = Path(os.getenv(
    'UNITY_TEST_PROJECT_PATH',
    r'D:\Unity\Unity_Project\CodingAgentTest',
)).resolve()
context_path = day06_path / 'memory' / 'project_context.json'
graph_path = day06_path / 'memory' / 'dependency_graph.json'
print('Unity 工程:', unity_project_path)
print('依赖图:', graph_path)

Unity 工程: D:\Unity\Unity_Project\CodingAgentTest
依赖图: D:\Anaconda\Project\AI-Coding-Agent\agent-learning\day06\memory\dependency_graph.json


## Steps

### 1. 扫描工程并生成依赖图

In [2]:
from memory.dependency_graph import DependencyGraphStore
from memory.project_context import ProjectContextStore
from tools.dependency_graph import DependencyGraphBuilder, DependencyGraphQuery
from tools.project_scanner import UnityProjectScanner
from workflow.project_understanding import ProjectUnderstandingNode

node = ProjectUnderstandingNode(
    UnityProjectScanner(unity_project_path),
    ProjectContextStore(context_path),
    DependencyGraphBuilder(),
    DependencyGraphStore(graph_path),
)
result = node.run({'agent_history': []})
if result['dependency_graph_status'] != 'success':
    raise RuntimeError(result['dependency_graph_error'])
dependency_graph = result['dependency_graph']
print(json.dumps(dependency_graph['summary'], ensure_ascii=False, indent=2))

[Project Understanding]扫描完成:D:\Anaconda\Project\AI-Coding-Agent\agent-learning\day06\memory\project_context.json
{
  "nodes": 21,
  "edges": 23,
  "types": 20,
  "assets": 1,
  "duplicate_types": 0,
  "ambiguous_references": 0,
  "source_errors": 0
}


### 2. 查看有界依赖边

边方向为“使用者 → 被依赖者”。这里只展示前 25 条，完整结果保存在 JSON 文件中。

In [3]:
edge_preview = dependency_graph['edges'][:25]
print(json.dumps(edge_preview, ensure_ascii=False, indent=2))

[
  {
    "source": "type:Game.Inventory.InventorySlotView",
    "target": "type:Game.Inventory.InventoryItemData",
    "kind": "type_reference",
    "evidence": {
      "path": "Assets/Generated/InventoryView.cs"
    }
  },
  {
    "source": "type:Game.Inventory.InventoryView",
    "target": "type:Game.Inventory.InventoryItemData",
    "kind": "type_reference",
    "evidence": {
      "path": "Assets/Generated/InventoryView.cs"
    }
  },
  {
    "source": "type:Game.Inventory.InventoryView",
    "target": "type:Game.Inventory.InventorySlotView",
    "kind": "type_reference",
    "evidence": {
      "path": "Assets/Generated/InventoryView.cs"
    }
  },
  {
    "source": "type:InventoryController",
    "target": "type:InventorySystem.InventoryItem",
    "kind": "type_reference",
    "evidence": {
      "path": "Assets/Generated/InventoryController.cs"
    }
  },
  {
    "source": "type:InventoryController",
    "target": "type:InventorySystem.InventoryManager",
    "kind": "type_refer

### 3. 查询直接、反向与传递依赖

In [4]:
graph_query = DependencyGraphQuery(dependency_graph)
controller_id = 'type:InventoryController'
manager_id = 'type:InventorySystem.InventoryManager'

query_result = {
    'controller_direct_dependencies': graph_query.dependencies(controller_id),
    'manager_direct_dependents': graph_query.dependents(manager_id),
    'controller_transitive_dependencies': graph_query.dependencies(
        controller_id, transitive=True
    ),
}
print(json.dumps(query_result, ensure_ascii=False, indent=2))

{
  "controller_direct_dependencies": [
    "type:InventorySystem.InventoryItem",
    "type:InventorySystem.InventoryManager"
  ],
  "manager_direct_dependents": [
    "type:InventoryController"
  ],
  "controller_transitive_dependencies": [
    "type:InventorySystem.InventoryItem",
    "type:InventorySystem.InventoryManager",
    "type:InventorySystem.InventorySaveData",
    "type:InventorySystem.InventorySlot",
    "type:InventorySystem.InventorySortType",
    "type:InventorySystem.ItemData",
    "type:InventorySystem.SlotSaveData"
  ]
}


## Checks

验证真实工程的核心库存依赖、持久化结果和诊断状态。

In [5]:
loaded_graph = DependencyGraphStore(graph_path).load()
summary = loaded_graph['summary']
assert loaded_graph['schema_version'] == 1
assert loaded_graph['project']['name'] == 'CodingAgentTest'
assert summary['types'] >= 1
assert summary['edges'] >= 1
assert summary['duplicate_types'] == 0
assert summary['ambiguous_references'] == 0
assert summary['source_errors'] == 0
assert manager_id in graph_query.dependencies(controller_id)
assert controller_id in graph_query.dependents(manager_id)
assert graph_path.is_file()
print('PASS: Day08 真实 Unity Dependency Graph 集成验收通过')

PASS: Day08 真实 Unity Dependency Graph 集成验收通过


## Next Steps

- Architecture 和 File Planner 已可使用依赖图判断修改影响范围。
- 当前解析后端聚焦项目内已知类型；如后续需要方法调用级语义，可保持 JSON 协议不变并替换为 Roslyn。
- 下一阶段进入 Day09 Test Agent。